In [1]:
!pip install -q openai

In [2]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_GROQ_API_KEY",
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)

# Test reviews and ground truth
reviews = [
    ("The battery life is amazing!", "Positive"),
    ("It works, nothing special.", "Neutral"),
    ("The screen is too small.", "Negative"),
    ("Excellent camera and fast performance.", "Positive"),
    ("The product arrived late and damaged.", "Negative")
]

# Zero-shot prompting
def zero_shot(review):
    prompt = f"""
Classify the following review as Positive, Neutral, or Negative.

Review: "{review}"

Answer with only one word:
Positive
Neutral
Negative
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()


# Few-shot prompting
def few_shot(review):
    prompt = f"""
Classify each review as Positive, Neutral, or Negative.

Review: "Best phone ever!"
Answer: Positive

Review: "The camera is slow."
Answer: Negative

Review: "Average quality, nothing special."
Answer: Neutral

Review: "{review}"
Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content.strip()


zero_correct = 0
few_correct = 0

print("=" * 95)
print(f"{'Review':45} {'Ground Truth':12} {'Zero-Shot':12} {'Few-Shot':12}")
print("=" * 95)

for review, truth in reviews:

    zero = zero_shot(review)
    few = few_shot(review)

    if zero.lower() == truth.lower():
        zero_correct += 1

    if few.lower() == truth.lower():
        few_correct += 1

    print(f"{review[:43]:45} {truth:12} {zero:12} {few:12}")

print("=" * 95)

print("\nAccuracy")
print(f"Zero-Shot : {zero_correct}/{len(reviews)}")
print(f"Few-Shot  : {few_correct}/{len(reviews)}")

print("\nConsistency")
if few_correct > zero_correct:
    print("Few-shot prompting performed better.")
elif zero_correct > few_correct:
    print("Zero-shot prompting performed better.")
else:
    print("Both methods performed equally well.")

print("\nFormat Adherence")
print("Outputs are expected to be only: Positive, Neutral, or Negative.")

Review                                        Ground Truth Zero-Shot    Few-Shot    
The battery life is amazing!                  Positive     Positive     Positive.   
It works, nothing special.                    Neutral      Neutral      Neutral.

The reviewer states that the product "works", which implies that it functions as intended, but they also mention that it's "nothing special", indicating a lack of enthusiasm or strong feelings either way. This suggests a neutral opinion.
The screen is too small.                      Negative     Negative     Negative.   
Excellent camera and fast performance.        Positive     Positive     Positive.   
The product arrived late and damaged.         Negative     Negative     Negative.   

Accuracy
Zero-Shot : 5/5
Few-Shot  : 0/5

Consistency
Zero-shot prompting performed better.

Format Adherence
Outputs are expected to be only: Positive, Neutral, or Negative.
